# 00 · Join UBIGEO y carga de fuentes de datos

Notebook de la Etapa 0 del pipeline: antes de tocar una sola imagen, se cargan y verifican todas las fuentes de datos y se construye la tabla maestra unida por UBIGEO.

## Setup

In [19]:
import pandas as pd
import geopandas as gpd
import requests

pd.set_option("display.max_columns", 50)


# Polígonos y ubicación geográfica

## Límite distrital INEI 2025 (shapefile)

Fuente: https://www.geogpsperu.com/2019/05/limite-distrital-actualizado-inei.html

Descarga manual (no tiene link directo de descarga)

In [20]:
# Ruta local al shapefile ya descargado
ruta_limite_distrital = "../data/raw/Limite Distrital INEI 2025 CPV/Limite Distrital INEI 2025 CPV.shp"

limite_distrital = gpd.read_file(ruta_limite_distrital)
print(limite_distrital.shape)
limite_distrital.head()


(1891, 10)


,UBIGEO,CCDD,CCPP,CCDI,DEPARTAMEN,PROVINCIA,DISTRITO,OBJECTID,ESRI_OID,geometry
0,010101,01,01,01,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,1.0,5.0,"POLYGON ((-77.8858 -6.1778, -77.88323 -6.17846..."
1,010102,01,01,02,AMAZONAS,CHACHAPOYAS,ASUNCION,2.0,6.0,"POLYGON ((-77.74482 -5.94497, -77.74482 -5.945..."
2,010103,01,01,03,AMAZONAS,CHACHAPOYAS,BALSAS,3.0,7.0,"POLYGON ((-77.9358 -6.69039, -77.93531 -6.6909..."
3,010104,01,01,04,AMAZONAS,CHACHAPOYAS,CHETO,4.0,8.0,"POLYGON ((-77.71486 -6.24598, -77.71485 -6.245..."
4,010105,01,01,05,AMAZONAS,CHACHAPOYAS,CHILIQUIN,5.0,9.0,"POLYGON ((-77.77405 -5.99598, -77.77328 -5.996..."


## Tabla de UBIGEO (crosswalk departamento-provincia-distrito)

Fuente (CSV público, se puede leer directo desde la URL):
https://raw.githubusercontent.com/jmcastagnetto/ubigeo-peru-aumentado/main/ubigeo_distrito.csv

In [21]:
url_ubigeo = "https://raw.githubusercontent.com/jmcastagnetto/ubigeo-peru-aumentado/main/ubigeo_distrito.csv"

ubigeo = pd.read_csv(url_ubigeo, dtype={"inei": str, "reniec": str})

print(ubigeo.shape)
ubigeo.head()

URLError: <urlopen error [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028)>

Como tenemos diferente cantidad de filas, verificaremos cuáles sonla que faltan en la 1ra

In [ ]:
fila_nan = ubigeo[ubigeo["inei"].isna()]
fila_nan

,inei,reniec,departamento,provincia,distrito,region,macroregion_inei,macroregion_minsa,iso_3166_2,fips,capital,superficie,pob_densidad_2020,altitude,latitude,longitude,indice_vulnerabilidad_alimentaria,idh_2019,pct_pobreza_total,pct_pobreza_extrema
1892,NaN,170107,MOQUEGUA,MARISCAL NIETO,SAN ANTONIO,MOQUEGUA,SUR,MACROREGION SUR,PE-MOQ,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Quitar la fila con inei vacío (San Antonio, Moquegua - dato incompleto)
ubigeo = ubigeo[ubigeo["inei"].notna()].copy()
print("Distritos después de quitar el nan:", ubigeo.shape[0])

Distritos después de quitar el nan: 1892


In [ ]:
# Los que están en el shapefile pero no en la tabla UBIGEO
print("--- En shapefile, no en tabla ---")
print(limite_distrital[limite_distrital["UBIGEO"].isin(["180107", "130112"])][["UBIGEO", "DEPARTAMEN", "PROVINCIA", "DISTRITO"]])

print()

# Los que están en la tabla UBIGEO pero no en el shapefile
print("--- En tabla, no en shapefile ---")
print(ubigeo[ubigeo["inei"].isin(["160109", "150144", "160114"])][["inei", "departamento", "provincia", "distrito"]])

--- En shapefile, no en tabla ---
      UBIGEO   DEPARTAMEN       PROVINCIA       DISTRITO
1182  130112  LA LIBERTAD        TRUJILLO  ALTO TRUJILLO
1534  180107     MOQUEGUA  MARISCAL NIETO    SAN ANTONIO

--- En tabla, no en shapefile ---
        inei departamento provincia                 distrito
1335  150144         LIMA      LIMA  SANTA MARIA DE HUACHIPA
1472  160109       LORETO    MAYNAS                 PUTUMAYO
1476  160114       LORETO    MAYNAS  TENIENTE MANUEL CLAVERO


In [ ]:
ubigeos_finales = set(limite_distrital["UBIGEO"].astype(str)) & set(ubigeo["inei"].astype(str))
print("Distritos finales para el análisis:", len(ubigeos_finales))

limite_distrital = limite_distrital[limite_distrital["UBIGEO"].isin(ubigeos_finales)].copy()
ubigeo = ubigeo[ubigeo["inei"].isin(ubigeos_finales)].copy()

print("Shapefile final:", limite_distrital.shape[0])
print("Tabla ubigeo final:", ubigeo.shape[0])

Distritos finales para el análisis: 1889
Shapefile final: 1889
Tabla ubigeo final: 1889


# Imágenes y variables satelitales (Google Earth Engine)

## Inicializar Earth Engine

Requiere cuenta de Google Earth Engine aprobada: https://earthengine.google.com

In [24]:
import ee

# NOTA: "viirs-peru" es el ID del proyecto de Google Cloud vinculado a Earth Engine, el nombre fue ese pero es para todo EARTH ENGINE
# sirve como autenticación para CUALQUIER dataset de Earth Engine (WorldCover, SRTM,
# Sentinel-2, Open Buildings, etc.), no solo para VIIRS.

try:
    ee.Initialize(project="viirs-peru")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="viirs-peru")

print("Google Earth Engine listo")


Google Earth Engine listo


### SRTM — Elevación del terreno (`USGS/SRTMGL1_003`)

**Qué es**
El SRTM (Shuttle Radar Topography Mission) es un modelo de elevación digital (DEM) global, generado por NASA/USGS a partir de una misión de radar en el año 2000. Cubre casi todo el planeta a una resolución espacial de ~30 metros por píxel. No es una serie temporal: es una sola "foto" de la topografía, porque el terreno no cambia año a año (a diferencia del gasto o la anemia, que sí varían).

**Qué se puede encontrar ahí**
- **Elevación (msnm)** por píxel — es la única variable nativa del dataset.
- A partir de la elevación se puede derivar **pendiente (slope)** con `ee.Terrain.slope()`, que mide qué tan inclinado es el terreno en cada punto (0° = plano, valores altos = muy empinado).

**Por qué es relevante para anemia**
No es un proxy indirecto — es directamente relevante por una razón clínica concreta: el protocolo del MINSA (siguiendo la guía de la OMS) **ajusta el punto de corte de hemoglobina para diagnosticar anemia según la altitud**, porque la hemoglobina sube naturalmente en zonas de mayor altura como mecanismo de adaptación a la menor presión de oxígeno. Sin controlar por altitud, el modelo podría confundir un efecto fisiológico normal con una diferencia real de prevalencia entre distritos.

Además, la pendiente del terreno es un buen proxy de accesibilidad: distritos con pendientes altas suelen tener viviendas más dispersas, caminos más difíciles y menor cobertura de seguimiento domiciliario para los programas de suplementación de hierro.

**Qué conviene extraer**
Por cada distrito (usando el polígono de `limite_distrital` con `reduceRegions`), dos estadísticos:

| Variable | Cómo se calcula | Para qué sirve |
|---|---|---|
| `elevacion_media` | Promedio de elevación (msnm) dentro del polígono distrital | Ajustar/controlar el punto de corte de hemoglobina por altitud |
| `pendiente_media` | Promedio de pendiente (°) dentro del polígono, derivado con `ee.Terrain.slope()` | Proxy de accesibilidad y dispersión del territorio |

**¿Por qué no hace falta más que esto?**

El SRTM no tiene versión por año — proviene de una única misión de radar realizada en el año 2000, y esa es la única medición que existe. A diferencia de Sentinel-2 (que sí tiene imágenes nuevas cada pocos días) o VIIRS (que sí tiene composites mensuales), no hay un "SRTM 2021" ni un "SRTM actual" que se pueda pedir en Earth Engine: el dataset es fijo, y así seguirá siendo salvo que la NASA lance una nueva misión de topografía.

Esto convierte a `elevacion_media` y `pendiente_media` en lo que en un panel de datos se llama una **variable time-invariant** (invariante en el tiempo): una característica del lugar, no del año. En la práctica, esto significa que en el join final estas dos columnas se unen solo por `ubigeo`, sin incluir `Año` en el merge — y su valor se repite igual en las cinco filas (2021 a 2025) de un mismo distrito, porque el terreno de un distrito no cambia de un año a otro. No es una redundancia ni un error: es exactamente cómo ya se comportan otras variables estáticas del proyecto, como `idh_2019` o `pct_pobreza_total` en la tabla `ubigeo`, que tampoco tienen una versión por año y también se repetirán igual en el join.

Por eso no hace falta pedir la elevación por año ni preocuparse por qué año usar: no existe esa variación en la fuente, así que un solo valor por distrito alcanza para las dos funciones que cumple esta variable (ajuste clínico del diagnóstico y proxy de accesibilidad). Si más adelante se necesita capturar heterogeneidad interna del distrito (ej. un distrito con zonas muy planas y muy empinadas a la vez), se puede agregar la desviación estándar de elevación como variable adicional — pero seguiría siendo una sola medición fija, no una serie por año.

### Cargar límite distrital como FeatureCollection de Earth Engine

Hasta ahora `limite_distrital` es un GeoDataFrame local (para hacer joins con pandas/geopandas).
Para calcular estadísticos zonales (`reduceRegions`) Earth Engine necesita esos mismos polígonos
como `ee.FeatureCollection`. Se sube una sola vez y se reutiliza para SRTM, y luego para las
demás fuentes de GEE (JRC Water, WorldCover, Sentinel-2).

In [25]:
# Filtramos SOLO San Juan de Lurigancho de tu GeoDataFrame local (antes de subir nada)
# UBIGEO de San Juan de Lurigancho: 150132 (Lima, provincia Lima, distrito SJL)
sjl = limite_distrital[limite_distrital["UBIGEO"] == "150132"]

print(sjl[["UBIGEO", "DEPARTAMEN", "PROVINCIA", "DISTRITO"]])

# Como es un solo polígono, el payload es chiquito — no hace falta simplificar geometría
sjl_ee = geemap.geopandas_to_ee(sjl[["UBIGEO", "geometry"]])
print("Polígonos subidos:", sjl_ee.size().getInfo())  # debería salir 1


# Imagen de elevación
srtm = ee.Image("USGS/SRTMGL1_003")

# Pendiente derivada de la elevación (en grados)
pendiente = ee.Terrain.slope(srtm)

      UBIGEO DEPARTAMEN PROVINCIA                DISTRITO
1324  150132       LIMA      LIMA  SAN JUAN DE LURIGANCHO
Polígonos subidos: 1


In [26]:
Map = geemap.Map()

paleta_elevacion = ["006633", "e5ffcc", "662a00", "d8d8d8", "f5f5f5"]
vis_elevacion = {"min": 0, "max": 1200, "palette": paleta_elevacion}  # SJL no es zona alta, bajamos el max

paleta_pendiente = ["00ff00", "ffff00", "ff0000"]
vis_pendiente = {"min": 0, "max": 40, "palette": paleta_pendiente}

Map.addLayer(srtm.clip(sjl_ee), vis_elevacion, "Elevación (SRTM)")
Map.addLayer(pendiente.clip(sjl_ee), vis_pendiente, "Pendiente")
Map.addLayer(sjl_ee.style(color="black", fillColor="00000000", width=2), {}, "Límite SJL")

Map.centerObject(sjl_ee, 12)  # hace zoom automático a la forma del distrito
Map

Map(center=[-11.945941584955625, -76.9714572042048], controls=(WidgetControl(options=['position', 'transparent…

In [27]:
srtm_pendiente = srtm.rename("elevacion").addBands(pendiente.rename("pendiente"))

sjl_zonal = srtm_pendiente.reduceRegions(
    collection=sjl_ee,
    reducer=ee.Reducer.mean(),
    scale=30,
)

# En vez de geemap.ee_to_df(sjl_zonal), bajamos el resultado nosotros mismos
info = sjl_zonal.getInfo()

filas = [feature["properties"] for feature in info["features"]]
sjl_resultado = pd.DataFrame(filas)
sjl_resultado

,UBIGEO,elevacion,pendiente
0,150132,650.93819,16.907267


### SRTM — elevación y pendiente media por distrito (todos los distritos del Perú)

**Qué se hace en esta etapa**

Se calcula, para cada uno de los 1889 distritos del Perú, un único valor resumen de elevación y
un único valor resumen de pendiente del terreno. El resultado es una tabla con una fila por
distrito — no una imagen, no un mapa: una tabla lista para unirse (por `ubigeo`) al resto de
variables del proyecto.

**De dónde sale el dato**

La fuente es el SRTM (`USGS/SRTMGL1_003`), un modelo de elevación digital de NASA/USGS a 30
metros de resolución. Es una sola medición fija del año 2000 — no existe una versión "por año",
así que este valor se calcula una sola vez y se repite igual en las cinco filas (2021–2025) de
cada distrito en el panel final, porque el terreno de un lugar no cambia de un año a otro (ver
nota sobre variables *time-invariant*).

**Qué son las "capas" en este contexto**

Cada fuente de datos de Earth Engine (elevación, pendiente, cobertura de suelo, agua, etc.) es
una *capa* distinta: una imagen independiente, del tamaño de todo el planeta, donde cada píxel
tiene un valor propio de esa variable. Elevación y pendiente son dos capas separadas — la
pendiente no viene incluida en el SRTM original, se calcula matemáticamente a partir de la
elevación con `ee.Terrain.slope()`, y después ambas capas se apilan en una sola imagen de dos
bandas para poder procesarlas juntas en un solo paso.

**Cómo se pasa de "millones de píxeles" a "un solo número por distrito": el reducer**

El SRTM tiene un píxel cada 30×30 metros. Un distrito de tamaño mediano puede contener decenas
o cientos de miles de esos píxeles, cada uno con su propio valor de elevación y de pendiente
(es justo la variación de colores que se ve al visualizar el mapa: verde donde el valor es bajo,
rojo donde es alto).

Para convertir todos esos píxeles en un solo número representativo del distrito, se usa un
**reducer** — la operación de Earth Engine que resume muchos valores en uno solo. El reducer
usado acá es `ee.Reducer.mean()`, que simplemente **promedia** todos los valores de los píxeles
que caen dentro del polígono de cada distrito. Es el mismo cálculo que haría `columna.mean()`
en pandas, con la diferencia de que aquí "la columna" son todos los píxeles contenidos dentro
de la forma geográfica del distrito, no filas de una tabla.

El promedio es la elección correcta para estas dos variables específicas: para elevación, porque
el ajuste clínico del punto de corte de hemoglobina por altitud (protocolo MINSA/OMS) se define
en función de la altitud típica de la población, no del punto más alto ni más bajo del distrito;
y para pendiente, porque sirve como una medida general de qué tan accidentado es el distrito en
conjunto, útil como proxy de accesibilidad.

**Un límite conocido del promedio**

En distritos con mucha variación interna — por ejemplo, uno con un valle plano y cerros
empinados a la vez — el promedio puede terminar representando un punto intermedio que no
describe bien a ninguna de las dos zonas reales del distrito. Esto se documenta como limitación
conocida y queda como posible extensión (agregar la desviación estándar de pendiente como
variable adicional, que mediría qué tan mixto es el terreno), sin ser indispensable para el
núcleo mínimo demostrable del proyecto.

**Resultado esperado**

Una tabla (`srtm_distrital`) de 1889 filas × 3 columnas: `ubigeo`, `elevacion_media` y
`pendiente_media`, lista para unirse al resto del maestro distrital solo por `ubigeo` (sin año).

In [28]:
import json
import pandas as pd

# ─────────────────────────────────────────────────────────────
# SRTM — elevación y pendiente media por distrito (por lotes)
# ─────────────────────────────────────────────────────────────

# Imagen de elevación + pendiente derivada, apiladas en una sola imagen
srtm_elevation = ee.Image("USGS/SRTMGL1_003").select("elevation")
srtm_slope = ee.Terrain.slope(srtm_elevation).rename("slope")
srtm_img = srtm_elevation.rename("elevation").addBands(srtm_slope)

# Base de distritos: solo ubigeo + geometry, geometría simplificada para no
# exceder el límite de payload de Earth Engine
distritos_base = (
    limite_distrital[["UBIGEO", "geometry"]]
    .rename(columns={"UBIGEO": "ubigeo"})
    .copy()
)
distritos_base["geometry"] = distritos_base.geometry.simplify(
    tolerance=0.005, preserve_topology=True
)
distritos_base = distritos_base[
    distritos_base.geometry.notna() & ~distritos_base.geometry.is_empty
].copy()

print("Distritos a procesar:", len(distritos_base))

# Solo promedio — max/min no aportan al objetivo del proyecto (ver nota abajo)
reducer_srtm = ee.Reducer.mean()

chunk_size = 40 # partimos en 40 para q no exceda el límite de payload de Earth Engine (aprox 1MB por request)
resultados = []

for i in range(0, len(distritos_base), chunk_size):
    chunk = distritos_base.iloc[i:i + chunk_size].copy()
    print(f"Procesando distritos {i + 1} a {min(i + chunk_size, len(distritos_base))}...")

    geojson_chunk = json.loads(chunk.to_json())
    distritos_ee_chunk = ee.FeatureCollection(geojson_chunk)

    stats_chunk = srtm_img.reduceRegions(
        collection=distritos_ee_chunk,
        reducer=reducer_srtm,
        scale=30,       # resolución nativa del SRTM
        tileScale=4,    # evita timeouts en distritos grandes
    )
    stats_chunk = stats_chunk.map(lambda f: f.setGeometry(None))

    features = stats_chunk.getInfo()["features"]
    df_chunk = pd.DataFrame([f["properties"] for f in features])
    resultados.append(df_chunk)

# Unir todos los lotes
srtm_distrital = pd.concat(resultados, ignore_index=True)
srtm_distrital["ubigeo"] = srtm_distrital["ubigeo"].astype(str).str.zfill(6)

# Renombrar a los nombres finales que se usarán en el maestro distrital
srtm_distrital = srtm_distrital.rename(columns={
    "elevation": "elevacion_media",
    "slope": "pendiente_media",
})[["ubigeo", "elevacion_media", "pendiente_media"]]

# Validación
print("\nSRTM distrital listo")
print("Distritos con SRTM:", len(srtm_distrital))
print("UBIGEO únicos:", srtm_distrital["ubigeo"].nunique())
print("Duplicados:", srtm_distrital["ubigeo"].duplicated().sum())

srtm_distrital.head()

Distritos a procesar: 1891
Procesando distritos 1 a 40...
Procesando distritos 41 a 80...
Procesando distritos 81 a 120...
Procesando distritos 121 a 160...
Procesando distritos 161 a 200...
Procesando distritos 201 a 240...
Procesando distritos 241 a 280...
Procesando distritos 281 a 320...
Procesando distritos 321 a 360...
Procesando distritos 361 a 400...
Procesando distritos 401 a 440...
Procesando distritos 441 a 480...
Procesando distritos 481 a 520...
Procesando distritos 521 a 560...
Procesando distritos 561 a 600...
Procesando distritos 601 a 640...
Procesando distritos 641 a 680...
Procesando distritos 681 a 720...
Procesando distritos 721 a 760...
Procesando distritos 761 a 800...
Procesando distritos 801 a 840...
Procesando distritos 841 a 880...
Procesando distritos 881 a 920...
Procesando distritos 921 a 960...
Procesando distritos 961 a 1000...
Procesando distritos 1001 a 1040...
Procesando distritos 1041 a 1080...
Procesando distritos 1081 a 1120...
Procesando distritos

,ubigeo,elevacion_media,pendiente_media
0,010101,2423.265986,21.252837
1,010102,2819.011116,19.819781
2,010103,2332.129471,30.776104
3,010104,2525.921415,20.679454
4,010105,2626.557058,22.795578


## Fuente: JRC Global Surface Water (JRC/GSW1_4/YearlyHistory)

**Qué es:** Clasificación anual de superficie de agua a 30m de resolución,
construida desde el archivo histórico de Landsat (1984-presente). Clasifica
cada píxel en: sin dato, no agua, agua estacional, agua permanente.

**Qué mide:** Presencia física de cuerpos de agua superficial. No mide agua
potable ni acceso a red de agua entubada.

**Por qué importa para anemia:** Proxy de dos factores del cuello de botella
de agua: (a) disponibilidad de agua para consumo informal o riego en zonas
sin red, y (b) exposición a fuentes de agua no tratada. Complementa —no
reemplaza— la variable de cuerpos de agua que saldrá de la segmentación en
Etapa 2: JRC da la serie histórica confiable a nivel distrital ahora;
la segmentación dará la ubicación exacta relativa a cada caserío.

**Variables extraídas:**
- `pct_agua_permanente` — fracción del área distrital con agua permanente
- `pct_agua_estacional` — fracción del área distrital con agua estacional

**Método de agregación:** Remapeo de la clase categórica a dos máscaras
binarias (permanente / estacional), luego `ee.Reducer.mean()` sobre cada
máscara → fracción de píxeles de esa clase sobre el total del polígono.

**Temporalidad:** Año único reciente (aprox. estable año a año en zona
rural, salvo eventos extremos). Une al maestro solo por `ubigeo`, igual
que SRTM.

**Fuente de polígonos:** `limite_distrital` (shapefile INEI 2025),
`reduceRegions` en lotes de 40 distritos.

## VIIRS Nighttime Lights

Dataset GEE: `NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG`

In [21]:
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG").select("avg_rad")
viirs_mean = viirs.mean()
print(viirs.first().getInfo())


{'type': 'Image', 'bands': [{'id': 'avg_rad', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [86400, 33600], 'crs': 'EPSG:4326', 'crs_transform': [0.0041666667, 0, -180.00208525335, 0, -0.0041666667, 75.00208393335001]}], 'version': 1494360354199000.0, 'id': 'NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG/20140101', 'properties': {'system:time_start': 1388534400000, 'system:footprint': {'type': 'LinearRing', 'coordinates': [[-180, -90], [180, -90], [180, 90], [-180, 90], [-180, -90]]}, 'system:time_end': 1391212800000, 'system:asset_size': 13625191512, 'system:index': '20140101'}}


## Sentinel-2 L2A

Dataset GEE: `COPERNICUS/S2_SR_HARMONIZED` — serie temporal, 12 bandas, 10m.

In [ ]:
sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2024-01-01", "2024-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
)
print("Número de imágenes:", sentinel2.size().getInfo())


## Planet NICFI

Registro gratuito para el trópico: https://www.planet.com/nicfi/

Requiere API key propia de Planet, no se accede vía Earth Engine público.

In [ ]:
planet_api_key = ""  # pegar aquí la API key de Planet NICFI

headers = {"Authorization": f"api-key {planet_api_key}"}
url_nicfi = "https://api.planet.com/basemaps/v1/mosaics"

# response = requests.get(url_nicfi, headers=headers)
# response.json()


## Google Open Buildings V3

Fuente: https://sites.research.google/gr/open-buildings/

Se puede acceder también como tabla en Earth Engine: `GOOGLE/Research/open-buildings/v3/polygons`

In [ ]:
open_buildings = ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons")
print(open_buildings.limit(5).getInfo())


# Datos administrativos y municipales

## RENAMU (Registro Nacional de Municipalidades)

**Fuente:** https://proyectos.inei.gob.pe/microdatos/ (buscar "RENAMU" — desde 2021 la encuesta se estandariza en un único módulo llamado "Registro Nacional de Municipalidades - RENAMU", sin división por módulos como en años anteriores)

RENAMU es la encuesta anual que aplica el INEI a todas las municipalidades del país. Acá se usa como fuente de variables de control para el modelo causal (Etapa 6, DML + Causal Forest): el riesgo que cubre es que el modelo confunda "el gasto no rinde porque el territorio no responde" con "el gasto no rinde porque la municipalidad gestiona mal". Sin este control, τ podría estar capturando capacidad de gestión municipal en vez de contexto territorial — que es justo lo que el proyecto busca aislar.

### Por qué el panel arranca en 2021 y no en 2020

- El módulo con la pregunta sobre anemia (P68_7) recién aparece en la encuesta 2021. RENAMU 2020 no la tiene, así que no sirve como punto de partida.
- Desde 2021 el formato está estandarizado en un único módulo: mismos nombres de columna año a año (confirmado revisando el portal directamente, carpeta por carpeta).
- Desde 2021 el propio CSV trae la columna `Ubigeo` ya limpia en la cabecera. En 2020 solo existe `idmunici`, y hay que reconstruir el ubigeo a mano con `.zfill(6)` — trabajo extra que no vale la pena si igual ese año no trae la variable que más importa.

Por eso el panel se arma como loop 2021 → último año disponible, con el mismo código para cada año, en vez de tratar 2020 como caso especial.

### Variables extraídas (panel por Ubigeo + Año)

| Variable | Código original | Nombre final | Redacción exacta del diccionario (RENAMU 2021) | Corrección de año |
|---|---|---|---|---|
| Personal municipal | P19D_T | `personal_total` | *"Total personal / 31 de diciembre 2020"* — dentro del bloque "Personal de la municipalidad, al 31 de diciembre 2020". Es un conteo, no Sí/No. | **Sí, retrospectiva.** El campo describe el personal al cierre del año anterior a la encuesta, no del año de la encuesta. Etiquetar como año = X−1 |
| Programa de prevención de anemia con MINSA | P68_7 | `programa_anemia` | *"En el año 2020, ¿La municipalidad implementó programas de control y prevención de la salud en coordinación con el MINSA en: Prevención y reducción de la anemia"* — ítem dentro de un checklist de 11 opciones (P68_1 a P68_11), no un Sí/No binario simple. Verificar valores únicos reales en el CSV antes de tratarla como booleana. | Sí, retrospectiva: encuesta año X pregunta por lo ejecutado en X−1. Etiquetar como año = X−1 |
| Centro de salud administrado por la municipalidad | P66_2 | `centro_salud_municipal` | *"¿En el Distrito funcionan establecimientos de salud administrados por la municipalidad: Centro de salud?"* — 1: Sí / 2: No | No. Pregunta en tiempo presente, sin referencia retrospectiva → el valor corresponde al mismo año de la encuesta |

### Corrección de año: las tres variables no llevan el mismo tratamiento

A diferencia de lo que se asumió inicialmente, **dos de las tres variables son retrospectivas, no solo una**:

- `personal_total` (P19D_T) y `programa_anemia` (P68_7) describen la situación del año **anterior** a la encuesta → se etiquetan con año = Año_encuesta − 1.
- `centro_salud_municipal` (P66_2) describe la situación **al momento de la encuesta** → se etiqueta con año = Año_encuesta, sin ajuste.

Si esta corrección no se aplica correctamente a cada variable, el merge con anemia (SIEN) y gasto (SIAF) queda desfasado, y el modelo terminaría comparando información municipal de un año con el gasto/anemia de otro año distinto.

### Pendiente antes de dar por cerrada la tabla

- Confirmar con `df["programa_anemia"].value_counts()` qué valores únicos trae realmente esa columna en el CSV cargado — el diccionario sugiere que es un ítem de checklist (0/Pase, 9/Sí), no un Sí/No de dos valores limpio como se pensaba.

In [34]:
from pathlib import Path
import pandas as pd

base_dir = Path("../data/raw/RENAMU")
años = range(2021, 2025)

columnas_necesarias = {
    "Ubigeo": "ubigeo",
    "P19D_T": "personal_total",
    "P68_7": "programa_anemia",
    "P66_2": "centro_salud_municipal",
}

nulos_renamu = ["#Â¡NULO!", "#¡NULO!"]

paneles = []

for año in años:
    carpeta = base_dir / str(año)

    if not carpeta.exists():
        print(f"⚠️ {año}: no existe la carpeta {carpeta}, saltando")
        continue

    csvs = [f for f in carpeta.iterdir() if f.suffix.lower() == ".csv"]

    if len(csvs) != 1:
        print(f"⚠️ {año}: encontré {len(csvs)} CSV en la carpeta, revisar manualmente")
        continue

    df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)

    faltantes = [c for c in columnas_necesarias if c not in df.columns]
    if faltantes:
        print(f"⚠️ {año}: faltan columnas {faltantes} — revisar nombre exacto en el diccionario {año}")
        print(f"   Columnas disponibles (primeras 15): {list(df.columns)[:15]}")
        continue

    df = df[list(columnas_necesarias.keys())].rename(columns=columnas_necesarias)
    df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)
    df["Año_encuesta"] = año

    # Diagnóstico rápido: confirmar qué valores trae programa_anemia antes de asumir Sí/No
    print(f"{año} — valores únicos en programa_anemia: {df['programa_anemia'].unique()}")

    paneles.append(df)
    print(f"✅ {año}: {df.shape[0]} municipalidades cargadas")

renamu_panel = pd.concat(paneles, ignore_index=True)

# --- Corrección de año: personal_total y programa_anemia son retrospectivas ---
# (describen el año anterior a la encuesta); centro_salud_municipal no lo es.
renamu_panel["Año_personal_total"] = renamu_panel["Año_encuesta"] - 1
renamu_panel["Año_programa_anemia"] = renamu_panel["Año_encuesta"] - 1
# centro_salud_municipal usa Año_encuesta directamente, sin columna adicional

print(renamu_panel.shape)
renamu_panel.head()

C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2925557397.py:31: DtypeWarning: Columns (0: P23_4_4_O, 1: P37A_5_O, 2: P73_4_O, 3: P95_5_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


2021 — valores únicos en programa_anemia: [0 7]
✅ 2021: 1874 municipalidades cargadas


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2925557397.py:31: DtypeWarning: Columns (0: P23_1_4_O, 1: P23_4_4_O, 2: P23_9_4_O, 3: P23_11_4_O, 4: P31_5_O, 5: P37A_5_O, 6: P73_4_O, 7: P79A_5_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


2022 — valores únicos en programa_anemia: [0 7]
✅ 2022: 1874 municipalidades cargadas


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2925557397.py:31: DtypeWarning: Columns (0: P23_14_1_O, 1: P23_14_2, 2: P28_5_O, 3: P37A_5_O, 4: P48_12_O, 5: P55_7_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


2023 — valores únicos en programa_anemia: [ 7.  0. nan]
✅ 2023: 1891 municipalidades cargadas
2024 — valores únicos en programa_anemia: [0 7]
✅ 2024: 1891 municipalidades cargadas
(7530, 7)


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2925557397.py:31: DtypeWarning: Columns (0: P23_13_2, 1: P25_4_O, 2: P28_5_O, 3: P48_12_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


,ubigeo,personal_total,programa_anemia,centro_salud_municipal,Año_encuesta,Año_personal_total,Año_programa_anemia
0,010101,208.0,0.0,2.0,2021,2020,2020
1,010102,1.0,0.0,2.0,2021,2020,2020
2,010103,7.0,0.0,2.0,2021,2020,2020
3,010104,5.0,7.0,2.0,2021,2020,2020
4,010105,2.0,0.0,2.0,2021,2020,2020


In [36]:
from pathlib import Path
import pandas as pd

base_dir = Path("../data/raw/RENAMU")
años = range(2021, 2025)  # ajustar según lo que confirmes disponible en el portal

columnas_necesarias = {
    "Ubigeo": "ubigeo",
    "P19D_T": "personal_total",
    "P68_7": "programa_anemia",
    "P66_2": "centro_salud_municipal",
}

nulos_renamu = ["#Â¡NULO!", "#¡NULO!"]

paneles = []

for año in años:
    carpeta = base_dir / str(año)
    if not carpeta.exists():
        continue

    csvs = [f for f in carpeta.iterdir() if f.suffix.lower() == ".csv"]
    if len(csvs) != 1:
        continue

    df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)

    if any(c not in df.columns for c in columnas_necesarias):
        continue

    df = df[list(columnas_necesarias.keys())].rename(columns=columnas_necesarias)
    df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)
    df["Año_encuesta"] = año

    paneles.append(df)

renamu_panel = pd.concat(paneles, ignore_index=True)

# --- Tres tablas filtradas, cada una con SOLO su variable y SU año real ---
# (listas para mergear después, cuando tengas tabla_maestra armada)

centro_salud = renamu_panel[["ubigeo", "Año_encuesta", "centro_salud_municipal"]].rename(
    columns={"Año_encuesta": "Año"}
)

personal = renamu_panel[["ubigeo", "Año_encuesta", "personal_total"]].copy()
personal["Año"] = personal["Año_encuesta"] - 1
personal = personal[["ubigeo", "Año", "personal_total"]]

anemia_muni = renamu_panel[["ubigeo", "Año_encuesta", "programa_anemia"]].copy()
anemia_muni["Año"] = anemia_muni["Año_encuesta"] - 1
anemia_muni = anemia_muni[["ubigeo", "Año", "programa_anemia"]]

print(renamu_panel.shape)
renamu_panel.head()

C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2681341985.py:27: DtypeWarning: Columns (0: P23_4_4_O, 1: P37A_5_O, 2: P73_4_O, 3: P95_5_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)
C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2681341985.py:27: DtypeWarning: Columns (0: P23_1_4_O, 1: P23_4_4_O, 2: P23_9_4_O, 3: P23_11_4_O, 4: P31_5_O, 5: P37A_5_O, 6: P73_4_O, 7: P79A_5_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)
C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2681341985.py:27: DtypeWarning: Columns (0: P23_14_1_O, 1: P23_14_2, 2: P28_5_O, 3: P37A_5_O, 4: P48_12_O, 5: P55_7_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


(7530, 5)


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2681341985.py:27: DtypeWarning: Columns (0: P23_13_2, 1: P25_4_O, 2: P28_5_O, 3: P48_12_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


,ubigeo,personal_total,programa_anemia,centro_salud_municipal,Año_encuesta
0,010101,208.0,0.0,2.0,2021
1,010102,1.0,0.0,2.0,2021
2,010103,7.0,0.0,2.0,2021
3,010104,5.0,7.0,2.0,2021
4,010105,2.0,0.0,2.0,2021


### Notas de codificación y tipos de dato

**`centro_salud_municipal` (P66_2):** es un indicador binario (Sí/No), no una cantidad. Corresponde específicamente al tipo de establecimiento "Centro de salud" administrado por la municipalidad — no incluye hospitales, postas, consultorios ni otros tipos (esos son preguntas separadas en el diccionario: P66_1, P66_3, P66_4, etc.). El campo de conteo real (`P66_2_1`, "Número de establecimientos") no forma parte de este panel.

**`programa_anemia` (P68_7):** el diccionario de RENAMU codifica esta pregunta como parte de un checklist de 11 programas de salud (P68_1 a P68_11), donde cada ítem usa su propia posición como código de "Sí" en vez de un 1/2 estándar. Para P68_7 específicamente: `0 = Pase` (no marcó esta opción) y `7 = Sí` (sí implementó el programa de anemia). Antes de usar esta variable en el modelo causal, se recodifica a booleano estándar (1 = Sí, 0 = No) para que no se interprete como una magnitud numérica.

**Tipos de dato:** las tres variables llegan como `float64` por los valores nulos (`NaN`) presentes en el CSV crudo. Se mantienen como `float64` hasta el merge final — convertir a `int` antes de imputar/tratar los nulos generaría un error, porque `NaN` no es representable como entero en pandas.

In [37]:
# --- Recodificación a booleano estándar (1 = Sí, 0 = No) ---

# programa_anemia: el diccionario de RENAMU codifica "Sí" como 7 (la posición
# del ítem "anemia" dentro del checklist P68_1 a P68_11), no como 1.
# Se recodifica para que el modelo no lo interprete como una magnitud.
renamu_panel["programa_anemia"] = (renamu_panel["programa_anemia"] == 7).astype("Int64")

# centro_salud_municipal: viene como 1=Sí, 2=No (estándar RENAMU).
# Se recodifica a 1=Sí, 0=No para mantener consistencia con programa_anemia.
renamu_panel["centro_salud_municipal"] = (renamu_panel["centro_salud_municipal"] == 1).astype("Int64")

# personal_total: es un conteo real (no booleano). Se pasa a entero nullable
# porque tiene NaN, y un int64 normal de numpy no admite nulos.
renamu_panel["personal_total"] = renamu_panel["personal_total"].astype("Int64")

# --- Reconstruir las tres tablas filtradas con los valores ya recodificados ---

centro_salud = renamu_panel[["ubigeo", "Año_encuesta", "centro_salud_municipal"]].rename(
    columns={"Año_encuesta": "Año"}
)

personal = renamu_panel[["ubigeo", "Año_encuesta", "personal_total"]].copy()
personal["Año"] = personal["Año_encuesta"] - 1
personal = personal[["ubigeo", "Año", "personal_total"]]

anemia_muni = renamu_panel[["ubigeo", "Año_encuesta", "programa_anemia"]].copy()
anemia_muni["Año"] = anemia_muni["Año_encuesta"] - 1
anemia_muni = anemia_muni[["ubigeo", "Año", "programa_anemia"]]

print(renamu_panel.dtypes)
renamu_panel.head()

ubigeo                      str
personal_total            Int64
programa_anemia           Int64
centro_salud_municipal    Int64
Año_encuesta              int64
dtype: object


,ubigeo,personal_total,programa_anemia,centro_salud_municipal,Año_encuesta
0,010101,208,0,0,2021
1,010102,1,0,0,2021
2,010103,7,0,0,2021
3,010104,5,1,0,2021
4,010105,2,0,0,2021


## ENAHO (Encuesta Nacional de Hogares)

Fuente: https://proyectos.inei.gob.pe/microdatos/ (buscar "ENAHO", módulos 1, 34, 200, 300, 400, 500)

In [ ]:
ruta_enaho_hogar = ""    # ej: "data/raw/ENAHO_1_hogar.csv"
ruta_enaho_sumaria = ""  # ej: "data/raw/ENAHO_34_sumaria.csv"

enaho_hogar = pd.read_csv(ruta_enaho_hogar)
enaho_sumaria = pd.read_csv(ruta_enaho_sumaria)

print(enaho_hogar.shape, enaho_sumaria.shape)
enaho_hogar.head()


## ENDES (Encuesta Demográfica y de Salud Familiar)

Fuente: https://proyectos.inei.gob.pe/microdatos/ (buscar "ENDES", módulos RECH0 y RECH23)

In [ ]:
ruta_endes_rech0 = ""   # ej: "data/raw/ENDES_1629_RECH0.csv"
ruta_endes_rech23 = ""  # ej: "data/raw/ENDES_1630_RECH23.csv"

endes_rech0 = pd.read_csv(ruta_endes_rech0)
endes_rech23 = pd.read_csv(ruta_endes_rech23)

print(endes_rech0.shape, endes_rech23.shape)
endes_rech0.head()


# Anemia (registro administrativo)

## SIEN / REUNIS (MINSA)

Fuente: https://www.minsa.gob.pe/reunis/

No tiene descarga directa en CSV — requiere revisar el tablero o solicitar la data en otro formato.

In [24]:
ruta_sien = "../data/raw/Minsa_reunis_anemia/Trama_Base_Anemia.xlsx"

sien = pd.read_excel(ruta_sien, dtype={"Ubigeo": str})
print(sien.shape)
sien.head()


(355876, 15)


,Año,Edad,Diresa,Departamento,Provincia,Distrito,Renipres,Ubigeo,Sexo,Evaluados,Anemia,Anemia Leve,Anemia Moderada,Anemia Severa,Normal
0,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5053.0,010202,M,12,0,0,0,0,12
1,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5053.0,010202,F,14,0,0,0,0,14
2,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5054.0,010202,M,3,0,0,0,0,3
3,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5054.0,010202,F,4,0,0,0,0,4
4,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5055.0,010202,M,6,0,0,0,0,6


In [28]:
# Filtrar solo menores de 3 años (Edad == 1), según confirmamos contra el tablero
sien_menores_3 = sien[sien["Edad"] == 1].copy()

anemia_distrital = (
    sien_menores_3
    .groupby(["ubigeo", "Año"], as_index=False)
    .agg(
        departamento=("Departamento", "first"),
        provincia=("Provincia", "first"),
        distrito=("Distrito", "first"),
        ninos_evaluados=("Evaluados", "sum"),
        ninos_con_anemia=("Anemia", "sum"),
        ninos_sin_anemia=("Normal", "sum"),
    )
)

anemia_distrital["prevalencia_anemia"] = (
    anemia_distrital["ninos_con_anemia"] / anemia_distrital["ninos_evaluados"]
)

print(anemia_distrital.shape)
anemia_distrital.head()

(13029, 9)


,ubigeo,Año,departamento,provincia,distrito,ninos_evaluados,ninos_con_anemia,ninos_sin_anemia,prevalencia_anemia
0,010101,2020,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,245,85,160,0.346939
1,010101,2021,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,234,71,163,0.303419
2,010101,2022,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,201,76,125,0.378109
3,010101,2023,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,837,160,677,0.191159
4,010101,2024,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,969,157,812,0.162023


In [ ]:
print(sorted(sien["Año"].unique()))
# tebemos datos desde el 2020

[np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


## Datos Abiertos — Anemia (MINSA / INS)

Fuente: https://datosabiertos.gob.pe/dataset/anemia

(No los tomamos porque simplemente reunen otras datas que ya se repitieron o son muy específicas, por ejemplo para ayacucho o por años)

In [30]:
# ruta_anemia_datosabiertos = ""  # ej: "data/raw/anemia_datosabiertos.csv"


# SIAF — Gasto Devengado: documentación del dataset

**Proyecto:** El Sol Que Más Rinde
**Fuente de datos:** Presupuesto y Ejecución de Gasto — Devengado Mensual, MEF
`datosabiertos.mef.gob.pe/dataset/presupuesto-y-ejecucion-de-gasto-devengado-mensual`

Este documento explica qué archivos descargamos, qué variables usamos de cada uno, qué significa cada una, y cómo las vamos a combinar para construir la tabla final `gasto_anemia` a nivel **ubigeo + Año**. La idea es que cualquiera del equipo pueda leer esto y entender el dataset sin tener que abrir el CSV.

---

## 1. Qué archivos descargamos y por qué

El MEF publica un CSV por año. La estructura cambia a partir de 2025:

| Años | Archivo | Estructura |
|---|---|---|
| 2021, 2022, 2023, 2024 | `YYYY-Gasto-Devengado.csv` | Un archivo por año, publicado como "anual" |
| 2025 | `2025-Gasto-Devengado-Mensual.csv` | Un archivo, pero con desglose mensual real |

Descargamos del 2021 al 2025 porque es el rango en el que también tenemos datos de anemia (SIEN) para hacer el cruce.

Existe también `2025-Gasto-Devengado-Diario.csv` (y su equivalente 2026), pero **no lo usamos**: es la misma información con otra frecuencia de actualización, no otra granularidad temporal útil para nosotros.

**Fase contable usada: Devengado.** El MEF reporta varias fases (Certificación, Compromiso, Devengado, Girado). Usamos Devengado porque es la fase en la que el bien o servicio ya fue recibido y la obligación de pago ya está reconocida — es el estándar para medir "cuánto se gastó realmente", no solo lo planeado ni lo pagado.

---

## 2. El problema de fondo: 2021–2024 no traen desglose mensual real

El diccionario oficial del MEF (`Gasto_Devengado_Diccionario.csv`) describe columnas mensuales (`MONTO_DEVENGADO_ENERO` … `MONTO_DEVENGADO_DICIEMBRE`) como si existieran en todos los años. En la práctica:

- **2021 a 2024:** las columnas mensuales vienen en cero. El único monto confiable es `MONTO_DEVENGADO_ANUAL`.
- **2025:** sí trae el desglose mensual real, poblado correctamente.

Como nuestra unidad de análisis es **distrito-año** (no distrito-mes), esto en realidad no nos genera un problema grave — solo tenemos que normalizar los dos formatos hacia un único número anual por fila.

### Cómo normalizamos

```
si ANO_EJE está entre 2021 y 2024:
    monto_devengado_anual = MONTO_DEVENGADO_ANUAL

si ANO_EJE == 2025:
    monto_devengado_anual = suma(MONTO_DEVENGADO_ENERO ... MONTO_DEVENGADO_DICIEMBRE)
```

Al final, todos los años quedan expresados en la misma columna: `monto_devengado_anual`, sin importar de qué archivo original vino la fila.

**Control de calidad sugerido:** para 2025, comparar la suma de los 12 meses contra `MONTO_DEVENGADO_ANUAL` (si ese campo también viene poblado ese año) — deberían coincidir. Si no coinciden, es señal de que hay que revisar el archivo antes de confiar en él.

---

## 3. Cómo se construye el UBIGEO

El CSV no trae un campo único de "ubigeo". Hay que armarlo concatenando tres códigos:

```
ubigeo = DEPARTAMENTO_EJECUTORA + PROVINCIA_EJECUTORA + DISTRITO_EJECUTORA
```

Cada uno es un código de 2 dígitos, así que el ubigeo final queda con 6 dígitos, igual que en RENAMU y en las demás tablas del proyecto (`centro_salud`, `personal`, `anemia_muni`).

⚠️ **Punto importante que hay que tener presente en todo el proyecto:**

Este ubigeo identifica **dónde está ubicada la entidad que ejecuta el gasto** (la unidad ejecutora), no necesariamente el distrito donde se presta el servicio. Por ejemplo, una unidad ejecutora de salud regional puede estar registrada en la capital de provincia y administrar gasto que en realidad se ejecuta en varios distritos rurales alrededor.

Esto es un riesgo ya identificado en el documento maestro del proyecto. Revisamos si existía una alternativa a nivel de la "meta" (la actividad/obra específica dentro del programa presupuestal), pero el campo de ubicación de la meta (`DEPARTAMENTO_META`) solo llega a nivel departamento, no distrito — así que no resuelve el problema.

**Decisión del equipo:** por ahora usamos `DISTRITO_EJECUTORA` como proxy del distrito de intervención, y lo dejamos declarado explícitamente en la pantalla de limitaciones de la aplicación final, tal como ya estaba planteado en el documento maestro.

---

## 4. Variables que sí usamos, y qué significa cada una

Nos quedamos solo con las variables directamente relacionadas con identificar el gasto en anemia y ubicarlo en el tiempo y el espacio. Todo lo demás (clasificación contable detallada, identificadores de la entidad, clasificación funcional, etc.) no aporta a nuestro nivel de análisis y lo descartamos.

| Variable | Qué significa | Para qué la usamos |
|---|---|---|
| `ANO_EJE` | Año de ejecución del presupuesto | Es la mitad de nuestra llave de unión (`ubigeo + Año`), igual que en RENAMU |
| `DEPARTAMENTO_EJECUTORA` | Código de departamento donde está ubicada la entidad que ejecuta el gasto | Primeros 2 dígitos del ubigeo |
| `PROVINCIA_EJECUTORA` | Código de provincia donde está ubicada la entidad | Dígitos 3-4 del ubigeo |
| `DISTRITO_EJECUTORA` | Código de distrito donde está ubicada la entidad | Dígitos 5-6 del ubigeo |
| `DISTRITO_EJECUTORA_NOMBRE` | Nombre del distrito | No entra al modelo — sirve solo para verificar visualmente que el ubigeo se armó bien y para detectar errores al momento de revisar el cruce |
| `PROGRAMA_PPTO` | Código del Programa Presupuestal | Con este código filtramos exactamente qué gasto es "gasto en anemia": PAN (0001) y los demás programas que toquen anemia (salud materno-neonatal, saneamiento rural, Cuna Más, Qali Warma, JUNTOS, incentivos municipales) |
| `PROGRAMA_PPTO_NOMBRE` | Nombre del programa presupuestal | Verificación de que filtramos el código correcto — también útil para poder decir en el pitch "de qué programa viene la plata" |
| `MONTO_DEVENGADO_ANUAL` | Monto total ejecutado en fase Devengado durante el año | Nuestra variable de gasto para 2021–2024 |
| `MONTO_DEVENGADO_ENERO` … `MONTO_DEVENGADO_DICIEMBRE` | Desglose mensual del gasto Devengado | Solo confiable en 2025 — se suman para construir el `monto_devengado_anual` equivalente de ese año |

**Variables que descartamos explícitamente** (identificadores de la entidad como `SECTOR`, `PLIEGO`, `EJECUTORA`; clasificación funcional como `FUNCION`, `DIVISION_FUNCIONAL`; clasificación contable como `GENERICA`, `SUBGENERICA`, `ESPECIFICA`; otras fases como `MONTO_CERTIFICADO_ANUAL`, `MONTO_COMPROMETIDO_ANUAL`, `MONTO_GIRADO_ANUAL`): no se relacionan directamente con identificar cuánto se gastó en anemia, dónde y cuándo — que es todo lo que necesita el modelo causal.

---

## 5. Cómo queda la tabla final

Después del filtro por `PROGRAMA_PPTO` (programas de anemia) y la normalización anual, agregamos por distrito-año, porque una misma combinación ubigeo-año puede tener varias filas (distintas entidades ejecutoras, distintas fuentes de financiamiento, etc. dentro del mismo programa):

```
gasto_anemia = groupby(['ubigeo', 'ANO_EJE'])['monto_devengado_anual'].sum()
```

Resultado esperado: una fila por distrito-año, con el monto total devengado en programas relacionados a anemia, lista para unirse con `centro_salud`, `personal` y `anemia_muni` usando `ubigeo + Año`.

---

## 6. Pendiente por resolver antes de cerrar esta etapa

- [ ] Confirmar la lista final y los códigos exactos de `PROGRAMA_PPTO` a incluir (PAN 0001 + complementarios)
- [ ] Correr el control de calidad de 2025 (suma de meses vs. anual, si aplica)
- [ ] Revisar cuántos distritos quedan con gasto = 0 después del join, como primera señal del tamaño del problema de `DISTRITO_EJECUTORA` vs. distrito de intervención real

In [42]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw/SIAF_gasto_devengado")

COLS_BASE = ['ANO_EJE', 'DEPARTAMENTO_EJECUTORA', 'PROVINCIA_EJECUTORA',
             'DISTRITO_EJECUTORA', 'DISTRITO_EJECUTORA_NOMBRE',
             'PROGRAMA_PPTO', 'PROGRAMA_PPTO_NOMBRE']

MESES = ['MONTO_DEVENGADO_ENERO','MONTO_DEVENGADO_FEBRERO','MONTO_DEVENGADO_MARZO',
         'MONTO_DEVENGADO_ABRIL','MONTO_DEVENGADO_MAYO','MONTO_DEVENGADO_JUNIO',
         'MONTO_DEVENGADO_JULIO','MONTO_DEVENGADO_AGOSTO','MONTO_DEVENGADO_SEPTIEMBRE',
         'MONTO_DEVENGADO_OCTUBRE','MONTO_DEVENGADO_NOVIEMBRE','MONTO_DEVENGADO_DICIEMBRE']

archivos = {
    2021: RAW_DIR / "2021-Gasto-Devengado" / "2021-Gasto-Devengado.csv",
    2022: RAW_DIR / "2022-Gasto-Devengado" / "2022-Gasto-Devengado.csv",
    2023: RAW_DIR / "2023-Gasto-Devengado" / "2023-Gasto-Devengado.csv",
    2024: RAW_DIR / "2024-Gasto-Devengado" / "2024-Gasto-Devengado.csv",
    2025: RAW_DIR / "2025-Gasto-Devengado-Mensual" / "2025-Gasto-Devengado-Mensual.csv",
}

In [43]:
dfs = []

for anio, ruta in archivos.items():
    cols = COLS_BASE + (['MONTO_DEVENGADO_ANUAL'] if anio <= 2024 else MESES)
    df = pd.read_csv(ruta, sep=",", dtype=str, usecols=cols)

    # ubigeo
    df['ubigeo'] = (df['DEPARTAMENTO_EJECUTORA'].str.zfill(2)
                     + df['PROVINCIA_EJECUTORA'].str.zfill(2)
                     + df['DISTRITO_EJECUTORA'].str.zfill(2))

    # monto anual: directo para 2021-2024, sumando meses para 2025
    if anio <= 2024:
        df['monto_devengado_anual'] = pd.to_numeric(df['MONTO_DEVENGADO_ANUAL'], errors='coerce')
    else:
        df['monto_devengado_anual'] = df[MESES].apply(pd.to_numeric, errors='coerce').sum(axis=1)

    dfs.append(df[['ANO_EJE', 'ubigeo', 'DISTRITO_EJECUTORA_NOMBRE',
                    'PROGRAMA_PPTO', 'PROGRAMA_PPTO_NOMBRE', 'monto_devengado_anual']])

gasto_siaf = pd.concat(dfs, ignore_index=True)
gasto_siaf['ANO_EJE'] = gasto_siaf['ANO_EJE'].astype(int)

gasto_siaf.head()

,ANO_EJE,ubigeo,DISTRITO_EJECUTORA_NOMBRE,PROGRAMA_PPTO,PROGRAMA_PPTO_NOMBRE,monto_devengado_anual
0,2021,150101,LIMA,9001,ACCIONES CENTRALES,0.00
1,2021,150101,LIMA,9001,ACCIONES CENTRALES,0.00
2,2021,150101,LIMA,9001,ACCIONES CENTRALES,0.00
3,2021,150101,LIMA,9001,ACCIONES CENTRALES,3129.80
4,2021,150101,LIMA,9001,ACCIONES CENTRALES,7923.24


Queremos validar q realmente el ubigeo anterior lo es porque lo habíamos construido

In [47]:
validacion = gasto_siaf[['ubigeo']].drop_duplicates().merge(
    inei[['UBIGEO']],  # <- reemplaza 'UBIGEO' por el nombre real que viste arriba
    left_on='ubigeo', right_on='UBIGEO', how='left', indicator=True
)

print(validacion['_merge'].value_counts())

no_cruzan = validacion.loc[validacion['_merge'] == 'left_only', 'ubigeo']
print(f"\n{len(no_cruzan)} ubigeos del SIAF no encontrados en INEI:")
print(no_cruzan.tolist())

_merge
both          1891
left_only        1
right_only       0
Name: count, dtype: int64

1 ubigeos del SIAF no encontrados en INEI:
['160405']


In [48]:
gasto_siaf[gasto_siaf['ubigeo'] == '160405'][['ubigeo', 'DISTRITO_EJECUTORA_NOMBRE']].drop_duplicates()

,ubigeo,DISTRITO_EJECUTORA_NOMBRE
11641131,160405,SANTA ROSA DE LORETO


Santa Rosa (a veces registrado como "Santa Rosa de Loreto") es un distrito de la provincia de Maynas, departamento de Loreto, creado en años recientes. Es exactamente el escenario que ya habían anotado en el documento maestro como riesgo conocido ("control de distritos creados después de 2017").

Validación del ubigeo contra INEI

Se construyó el ubigeo del SIAF (`DEPARTAMENTO_EJECUTORA + PROVINCIA_EJECUTORA + DISTRITO_EJECUTORA`)
y se cruzó contra el shapefile oficial de límites distritales de INEI (2025 CPV).

**Resultado: 1891 de 1892 ubigeos únicos (99.9%) cruzan correctamente.**

El único que no cruza es `160405` — **Santa Rosa de Loreto**, provincia de Mariscal Ramón Castilla,
departamento de Loreto. Este distrito fue creado el 3 de julio de 2025 (Ley N° 32403), separándose
del distrito de Yavarí. El shapefile de INEI usado como referencia fue generado antes de esa fecha,
por lo que no lo incluye.

**Decisión:** no es un error del pipeline ni del ubigeo construido — es un distrito legítimamente
nuevo que el shapefile de referencia todavía no contempla. Se documenta como limitación conocida.
Para efectos del modelo, el gasto de `160405` puede: (a) excluirse de la muestra por no tener
polígono ni contexto satelital propio, o (b) reasignarse temporalmente a Yavarí (su distrito de
origen) si el volumen de gasto es significativo. Pendiente decidir con el equipo.

In [ ]:
gasto_siaf[gasto_siaf['ubigeo'] == '160405']['monto_devengado_anual'].sum()
# veamos cuanto era el monto devengado en ese distrito q no toamremos en cuenta, es una cifra pequeña asi q no le haremos caso

np.float64(402358.7)

**Decisión del equipo:** se excluye `160405` (Santa Rosa de Loreto) de `gasto_siaf`.
Monto excluido: S/ 402,358.70 (2021-2025, todos los programas presupuestales) — marginal
frente al total nacional. Además, al no existir en el shapefile de INEI usado, tampoco
sería posible construir su contexto satelital para el modelo causal, así que conservar
su gasto no aportaría una observación utilizable de todas formas.

In [54]:
# cuales son los porgramas?
programas = gasto_siaf[['PROGRAMA_PPTO', 'PROGRAMA_PPTO_NOMBRE']].drop_duplicates().sort_values('PROGRAMA_PPTO')
print(programas)

        PROGRAMA_PPTO                               PROGRAMA_PPTO_NOMBRE
8240             0001                    PROGRAMA ARTICULADO NUTRICIONAL
8808             0002                             SALUD MATERNO NEONATAL
9102             0016                                       TBC-VIH/SIDA
7777             0017                ENFERMEDADES METAXENICAS Y ZOONOSIS
7888             0018                      ENFERMEDADES NO TRANSMISIBLES
...               ...                                                ...
7517403          0151  REDUCCION DE LA CORRUPCION EN EL USO DE LOS RE...
787459           1001  PRODUCTOS ESPECIFICOS PARA DESARROLLO INFANTIL...
6321             1002  PRODUCTOS ESPECIFICOS PARA REDUCCION DE LA VIO...
0                9001                                 ACCIONES CENTRALES
221              9002  ASIGNACIONES PRESUPUESTARIAS QUE NO RESULTAN E...

[93 rows x 2 columns]


In [55]:
programas.to_csv("../data/raw/SIAF_gasto_devengado/programas_presupuestales_unicos.csv",
                  index=False, sep=";")

print(f"Guardado: {len(programas)} programas")

Guardado: 93 programas


In [56]:
# Gasto total: todos los programas presupuestales, sin filtrar
gasto_total = (
    gasto_siaf
    .groupby(['ANO_EJE', 'ubigeo'], as_index=False)['monto_devengado_anual']
    .sum()
    .rename(columns={'monto_devengado_anual': 'gasto_total'})
)

# Gasto en anemia: solo PAN (0001)
gasto_anemia = (
    gasto_siaf[gasto_siaf['PROGRAMA_PPTO'] == '0001']
    .groupby(['ANO_EJE', 'ubigeo'], as_index=False)['monto_devengado_anual']
    .sum()
    .rename(columns={'monto_devengado_anual': 'gasto_anemia_pan'})
)

# Unimos ambos en una sola tabla, una fila por año-ubigeo
gasto_final = gasto_total.merge(gasto_anemia, on=['ANO_EJE', 'ubigeo'], how='left')

# Si un distrito no tuvo gasto en PAN ese año, el merge deja NaN — lo correcto es 0, no vacío
gasto_final['gasto_anemia_pan'] = gasto_final['gasto_anemia_pan'].fillna(0)

print(f"Filas (año-ubigeo): {len(gasto_final):,}")
gasto_final.head()

Filas (año-ubigeo): 9,453


,ANO_EJE,ubigeo,gasto_total,gasto_anemia_pan
0,2021,010101,7.452997e+08,44747568.78
1,2021,010102,9.765828e+05,0.00
2,2021,010103,1.303335e+06,16060.00
3,2021,010104,2.132187e+06,0.00
4,2021,010105,1.380533e+06,5584.48


In [57]:
# Cuántos distritos únicos hay por año
resumen_anual = gasto_final.groupby('ANO_EJE')['ubigeo'].nunique()
print(resumen_anual)

print(f"\nTotal filas: {len(gasto_final):,}")
print(f"Total distritos únicos (todos los años): {gasto_final['ubigeo'].nunique()}")

ANO_EJE
2021    1889
2022    1890
2023    1891
2024    1891
2025    1892
Name: ubigeo, dtype: int64

Total filas: 9,453
Total distritos únicos (todos los años): 1892


In [58]:
'160405' in gasto_final['ubigeo'].values

True

In [59]:
# 1. Excluir Santa Rosa de Loreto (creado julio 2025, no existe en el shapefile INEI de referencia)
n_antes = len(gasto_siaf)
monto_excluido = gasto_siaf.loc[gasto_siaf['ubigeo'] == '160405', 'monto_devengado_anual'].sum()

gasto_siaf = gasto_siaf[gasto_siaf['ubigeo'] != '160405'].copy()

print(f"Filas excluidas: {n_antes - len(gasto_siaf)}")
print(f"Monto excluido: S/ {monto_excluido:,.2f}")

# 2. Volver a generar gasto_total, gasto_anemia y el merge final
gasto_total = (
    gasto_siaf
    .groupby(['ANO_EJE', 'ubigeo'], as_index=False)['monto_devengado_anual']
    .sum()
    .rename(columns={'monto_devengado_anual': 'gasto_total'})
)

gasto_anemia = (
    gasto_siaf[gasto_siaf['PROGRAMA_PPTO'] == '0001']
    .groupby(['ANO_EJE', 'ubigeo'], as_index=False)['monto_devengado_anual']
    .sum()
    .rename(columns={'monto_devengado_anual': 'gasto_anemia_pan'})
)

gasto_final = gasto_total.merge(gasto_anemia, on=['ANO_EJE', 'ubigeo'], how='left')
gasto_final['gasto_anemia_pan'] = gasto_final['gasto_anemia_pan'].fillna(0)

# 3. Verificar que ya no está, y que el total de distritos bajó a 1891
print('160405' in gasto_final['ubigeo'].values)
print(f"Distritos únicos: {gasto_final['ubigeo'].nunique()}")
print(gasto_final.groupby('ANO_EJE')['ubigeo'].nunique())

Filas excluidas: 67
Monto excluido: S/ 402,358.70
False
Distritos únicos: 1891
ANO_EJE
2021    1889
2022    1890
2023    1891
2024    1891
2025    1891
Name: ubigeo, dtype: int64


In [60]:
# ¿Cuáles son los distritos que no aparecen en todos los años?
todos_los_ubigeos = gasto_final['ubigeo'].unique()
años = gasto_final['ANO_EJE'].unique()

panel_completo = pd.MultiIndex.from_product([años, todos_los_ubigeos], names=['ANO_EJE', 'ubigeo']).to_frame(index=False)

faltantes = panel_completo.merge(gasto_final[['ANO_EJE', 'ubigeo']], on=['ANO_EJE', 'ubigeo'], how='left', indicator=True)
faltantes = faltantes[faltantes['_merge'] == 'left_only']

print(faltantes)

      ANO_EJE  ubigeo     _merge
1889     2021  180107  left_only
1890     2021  130112  left_only
3781     2022  130112  left_only


# Join final por UBIGEO

Una vez cargadas todas las fuentes, se unen por la llave `ubigeo` para construir la tabla maestra distrital.

In [ ]:
# maestro_distritos = (
#     ubigeo
#     .merge(renamu, on="ubigeo", how="left")
#     .merge(enaho_sumaria, on="ubigeo", how="left")
#     .merge(endes_rech0, on="ubigeo", how="left")
#     .merge(sien, on="ubigeo", how="left")
#     .merge(anemia_da, on="ubigeo", how="left")
#     .merge(siaf, on="ubigeo", how="left")
# )
#
# print("Distritos en maestro:", maestro_distritos.shape[0])
# print("Duplicados UBIGEO:", maestro_distritos["ubigeo"].duplicated().sum())
# maestro_distritos.head()


## Guardar tabla maestra

In [ ]:
ruta_salida = ""  # ej: "data/clean/maestro_distritos_2025.csv"

# maestro_distritos.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
# print("Archivo guardado en:", ruta_salida)
